<a href="https://colab.research.google.com/github/Muhammad-Ahmad-1341661/deep-learning-models/blob/main/TEST_Ahmad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install Libraries
!pip install transformers torch pandas scikit-learn nltk gdown

import pandas as pd
import numpy as np
import ast
import re
import nltk
from transformers import pipeline
import os

# Define Google Drive file IDs extracted from your links
mv_file_id = '1jEa_TvRcfAw69UIBPd00DXjd3socPk3S' # Corrected: This ID should point to the actual TMDB movies dataset
cr_file_id = '1bC7WhnkmuTKWPnoLk639bcqs9AjGkwA2'
imdb_file_id = '11OMD-NHr110vXgDCra5ErSK-XD0etGKj' # Corrected: This ID should point to the actual IMDB dataset

# Filenames
mv_file = 'tmdb_5000_movies.csv'
cr_file = 'tmdb_5000_credits.csv'
imdb_data_file = 'IMDB_Dataset.csv'

# Download files from Google Drive using gdown
print("Downloading datasets from Google Drive... Please wait.")
!gdown --id {mv_file_id} -O {mv_file}
!gdown --id {cr_file_id} -O {cr_file}
!gdown --id {imdb_file_id} -O {imdb_data_file}

# Load Datasets
print("\nLoading datasets into memory...")
movies = pd.read_csv(mv_file)
credits = pd.read_csv(cr_file, on_bad_lines='skip') # Added on_bad_lines='skip'
# IMDB load karte waqt error handling (bad lines skip)
imdb_data = pd.read_csv(imdb_data_file, on_bad_lines='skip').sample(200, random_state=42)

# Debug: Print columns to verify 'title' column existence
print("Columns in 'movies' DataFrame before merge:", movies.columns.tolist())
print("Columns in 'credits' DataFrame before merge:", credits.columns.tolist())

# Merge TMDB
movies = movies.merge(credits, on='title')
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'popularity']]

print("\nDatasets Download aur Merge ho gaye hain ✅")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1jEa_TvRcfAw69UIBPd00DXjd3socPk3S
To: /content/tmdb_5000_movies.csv
100% 40.0M/40.0M [00:00<00:00, 104MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1bC7WhnkmuTKWPnoLk639bcqs9AjGkwA2
To: /content/tmdb_5000_credits.csv
100% 5.70M/5.70M [00:00<00:00, 43.2MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...


In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

def convert(obj):
    return [i['name'] for i in ast.literal_eval(obj)]

def convert_cast(obj):
    return [i['name'] for i in ast.literal_eval(obj)][:3]

def fetch_dir(obj):
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director': return [i['name']]
    return []

# Data Cleaning & Feature Engineering
movies.dropna(inplace=True)
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert_cast)
movies['crew'] = movies['crew'].apply(fetch_dir)

# Create "Feature Soup" for Recommendation
movies['tags'] = movies['overview'].apply(lambda x: x.split()) + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x).lower())

def clean_text(text):
    text = re.sub(r'<.*?>', '', text.lower()) # HTML removal
    text = re.sub(r'[^a-z]', ' ', text) # Symbols removal
    return text

print("Preprocessing aur Feature Soup mukammal ho gaya ✅")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Preprocessing aur Feature Soup mukammal ho gaya ✅


In [ ]:
print(f"\nTest Accuracy (on IMDb subset): {accuracy*100:.2f}%")
print("Note: This model (DistilBERT) is pre-trained, so there is no 'training accuracy' or 'gap' to report from this specific notebook's execution.")

In [ ]:
from sklearn.metrics import accuracy_score

print("BERT Model (DistilBERT) load ho raha hai... (Approx 1-2 mins)")
# Using DistilBERT for faster processing in Colab
bert_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Evaluate Accuracy on IMDB subset
print("Calculating BERT Model Accuracy on IMDb Test Data...")
imdb_data['cleaned'] = imdb_data['review'].apply(clean_text)
y_true = imdb_data['sentiment'].map({'positive': 1, 'negative': -1})

y_pred = []
for r in imdb_data['cleaned']:
    res = bert_analyzer(r[:512])[0] # BERT supports 512 tokens
    y_pred.append(1 if res['label'] == 'POSITIVE' else -1)

accuracy = accuracy_score(y_true, y_pred)
print(f"\n✅ BERT MODEL EVALUATION COMPLETED")
print(f"⭐ System Model Accuracy: {accuracy*100:.2f}%")

BERT Model (DistilBERT) load ho raha hai... (Approx 1-2 mins)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Calculating BERT Model Accuracy on IMDb Test Data...

✅ BERT MODEL EVALUATION COMPLETED
⭐ System Model Accuracy: 80.50%


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# TF-IDF for Movie Search/Recommendation
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['tags'])
similarity = cosine_similarity(tfidf_matrix)

def get_recommendations(movie_title):
    try:
        idx = movies[movies['title'].str.lower() == movie_title.lower()].index[0]
        distances = sorted(list(enumerate(similarity[idx])), reverse=True, key=lambda x: x[1])

        recs = []
        for i in distances[1:6]: # Get top 5 similar movies
            recs.append(movies.iloc[i[0]].title)
        return recs
    except:
        return None

def analyze_review_detailed(text):
    res = bert_analyzer(text[:512])[0]
    lbl, score = res['label'], res['score']

    # Detailed mapping based on BERT confidence score
    if lbl == 'POSITIVE':
        if score > 0.95: return "Excellent/Masterpiece 😍", "Must Watch"
        return "Good/Standard 😊", "Watchable"
    else:
        if score > 0.95: return "Very Bad/Waste of Time 🤮", "Do Not Recommend"
        return "Poor/Negative ☹️", "Skip It"

print("Recommendation & Sentiment Logic Engine Taiyar hai ✅")

Recommendation & Sentiment Logic Engine Taiyar hai ✅


In [ ]:
print("\n" + "="*60)
print("🎬 ADVANCED MOVIE SEARCH & BERT SENTIMENT SYSTEM 🎬")
print("="*60)

# Display Trending Movies automatically
print("\n🔥 PEOPLE MOST WATCHED (Trending Now):")
trending = movies.sort_values('popularity', ascending=False).head(5)
for i, row in trending.iterrows():
    print(f"   ⭐ {row['title']}")

# Interactive Loop
while True:
    print("\n" + "-"*50)
    query = input("Movie ka naam likhen search karne ke liye (ya 'exit' likhen): ").strip()

    if query.lower() == 'exit':
        print("System Band ho raha hai. Shoukriya! 🍿")
        break

    # 1. Search Result Logic
    matches = movies[movies['title'].str.contains(query, case=False, na=False)]

    if matches.empty:
        print("❌ Maaf kijiye! Movie nahi mili. Dobara sahi naam likhen.")
        continue

    selected_movie = matches.iloc[0]['title']
    print(f"\n✅ MOVIE FOUND: {selected_movie}")
    print(f"📝 STORY OVERVIEW: {matches.iloc[0]['overview'][:200]}...")

    # 2. Recommendations (Directly under search result)
    print("\n💡 SIMILAR MOVIES YOU MIGHT LIKE (Recommendations):")
    recs = get_recommendations(selected_movie)
    if recs:
        for r in recs: print(f"   👉 {r}")

    # 3. User Review for the Movie
    print("\n" + "."*40)
    user_review = input(f"Aapne agar '{selected_movie}' dekhi hai, to apna review den: ")

    if user_review:
        # Using BERT for deep analysis
        label, verdict = analyze_review_detailed(user_review)
        print(f"\n📊 BERT SENTIMENT ANALYSIS RESULT:")
        print(f"   Review Quality: {label}")
        print(f"   Final Recommendation: {verdict}")


🎬 ADVANCED MOVIE SEARCH & BERT SENTIMENT SYSTEM 🎬

🔥 PEOPLE MOST WATCHED (Trending Now):
   ⭐ Minions
   ⭐ Interstellar
   ⭐ Deadpool
   ⭐ Guardians of the Galaxy
   ⭐ Mad Max: Fury Road

--------------------------------------------------
Movie ka naam likhen search karne ke liye (ya 'exit' likhen): Minions

✅ MOVIE FOUND: Minions
📝 STORY OVERVIEW: Minions Stuart, Kevin and Bob are recruited by Scarlet Overkill, a super-villain who, alongside her inventor husband Herb, hatches a plot to take over the world....

💡 SIMILAR MOVIES YOU MIGHT LIKE (Recommendations):
   👉 Despicable Me 2
   👉 Despicable Me
   👉 Darling Companion
   👉 Austin Powers: The Spy Who Shagged Me
   👉 Million Dollar Arm

........................................
Aapne agar 'Minions' dekhi hai, to apna review den: very bad movie

📊 BERT SENTIMENT ANALYSIS RESULT:
   Review Quality: Very Bad/Waste of Time 🤮
   Final Recommendation: Do Not Recommend

--------------------------------------------------
Movie ka naam likhe

In [ ]:
from google.colab import drive
import os
import joblib # Import joblib for saving TF-IDF and similarity matrix
import pandas as pd # Import pandas for DataFrame operations

# 1. Google Drive mount karein
drive.mount('/content/drive')

# 2. Folder path define karein jahan model save karna hai
save_directory = "/content/drive/MyDrive/Movie_BERT_Model"

# Agar folder nahi bana hua to bana den
if not os.path.exists(save_directory):
    os.makedirs(save_directory)

# 3. Pehle se load shuda 'bert_analyzer' model aur tokenizer ko save karein
print(f"Model save ho raha hai Drive mein: {save_directory}")

# Access the underlying model and tokenizer from the pipeline and save them separately
bert_analyzer.model.save_pretrained(save_directory)
bert_analyzer.tokenizer.save_pretrained(save_directory)

# 4. Save TF-IDF Vectorizer, Similarity Matrix, and the preprocessed 'movies' DataFrame
joblib.dump(tfidf, os.path.join(save_directory, 'tfidf_vectorizer.joblib'))
joblib.dump(similarity, os.path.join(save_directory, 'similarity_matrix.joblib'))
movies.to_pickle(os.path.join(save_directory, 'movies_df.pkl')) # Save the movies DataFrame

print("Model and Recommendation Components successfully save ho gaye! ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model save ho raha hai Drive mein: /content/drive/MyDrive/Movie_BERT_Model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and Recommendation Components successfully save ho gaye! ✅


In [ ]:
from google.colab import drive
from transformers import pipeline
import os
import joblib # Import joblib for loading TF-IDF and similarity matrix
import pandas as pd # Import pandas for DataFrame operations

# 1. Google Drive mount karein
drive.mount('/content/drive')

# 2. Wohi path define karein jahan model save kiya tha
load_directory = "/content/drive/MyDrive/Movie_BERT_Model"

if os.path.exists(load_directory):
    print("Drive se sabhi components load ho rahe hain... Please wait.")

    # BERT Sentiment Pipeline ko Drive ke folder se load karein
    bert_analyzer = pipeline("sentiment-analysis", model=load_directory, tokenizer=load_directory)

    # Load TF-IDF Vectorizer, Similarity Matrix, and the preprocessed 'movies' DataFrame
    try:
        global tfidf, similarity, movies # Declare global to modify existing variables
        tfidf = joblib.load(os.path.join(load_directory, 'tfidf_vectorizer.joblib'))
        similarity = joblib.load(os.path.join(load_directory, 'similarity_matrix.joblib'))
        movies = pd.read_pickle(os.path.join(load_directory, 'movies_df.pkl')) # Load the movies DataFrame
        print("All Recommendation and Sentiment components (TF-IDF, Similarity, Movies DataFrame, BERT Model) successfully load ho gaye! ✅")
    except FileNotFoundError:
        print("❌ Error: Kuch components Drive mein nahi mile. Pehle save wala cell chalaen.")
        # Optionally, re-run the previous cells to regenerate if not found

    print("Sabhi modules Drive se successfully load ho gaye! 🚀")
else:
    print("❌ Error: Drive mein model folder nahi mila. Pehle save wala cell chalaen.")

print("Ab aap interactive system chala sakte hain!")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive se sabhi components load ho rahe hain... Please wait.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

All Recommendation and Sentiment components (TF-IDF, Similarity, Movies DataFrame, BERT Model) successfully load ho gaye! ✅
Sabhi modules Drive se successfully load ho gaye! 🚀
Ab aap interactive system chala sakte hain!


In [ ]:
print("\n" + "="*60)
print("🎬 ADVANCED MOVIE SEARCH & BERT SENTIMENT SYSTEM 🎬")
print("="*60)

# Display Trending Movies automatically
print("\n🔥 PEOPLE MOST WATCHED (Trending Now):")
trending = movies.sort_values('popularity', ascending=False).head(5)
for i, row in trending.iterrows():
    print(f"   ⭐ {row['title']}")

# Interactive Loop
while True:
    print("\n" + "-"*50)
    query = input("Movie ka naam likhen search karne ke liye (ya 'exit' likhen): ").strip()

    if query.lower() == 'exit':
        print("System Band ho raha hai. Shoukriya! 🍿")
        break

    # 1. Search Result Logic
    matches = movies[movies['title'].str.contains(query, case=False, na=False)]

    if matches.empty:
        print("❌ Maaf kijiye! Movie nahi mili. Dobara sahi naam likhen.")
        continue

    selected_movie = matches.iloc[0]['title']
    print(f"\n✅ MOVIE FOUND: {selected_movie}")
    print(f"📝 STORY OVERVIEW: {matches.iloc[0]['overview'][:200]}...")

    # 2. Recommendations (Directly under search result)
    print("\n💡 SIMILAR MOVIES YOU MIGHT LIKE (Recommendations):")
    recs = get_recommendations(selected_movie)
    if recs:
        for r in recs: print(f"   👉 {r}")

    # 3. User Review for the Movie
    print("\n" + "."*40)
    user_review = input(f"Aapne agar '{selected_movie}' dekhi hai, to apna review den: ")

    if user_review:
        # Using BERT for deep analysis
        label, verdict = analyze_review_detailed(user_review)
        print(f"\n📊 BERT SENTIMENT ANALYSIS RESULT:")
        print(f"   Review Quality: {label}")
        print(f"   Final Recommendation: {verdict}")


🎬 ADVANCED MOVIE SEARCH & BERT SENTIMENT SYSTEM 🎬

🔥 PEOPLE MOST WATCHED (Trending Now):
   ⭐ Minions
   ⭐ Interstellar
   ⭐ Deadpool
   ⭐ Guardians of the Galaxy
   ⭐ Mad Max: Fury Road

--------------------------------------------------
Movie ka naam likhen search karne ke liye (ya 'exit' likhen): Minions

✅ MOVIE FOUND: Minions
📝 STORY OVERVIEW: Minions Stuart, Kevin and Bob are recruited by Scarlet Overkill, a super-villain who, alongside her inventor husband Herb, hatches a plot to take over the world....

💡 SIMILAR MOVIES YOU MIGHT LIKE (Recommendations):
   👉 Despicable Me 2
   👉 Despicable Me
   👉 Darling Companion
   👉 Austin Powers: The Spy Who Shagged Me
   👉 Million Dollar Arm

........................................
Aapne agar 'Minions' dekhi hai, to apna review den: VERY FANTASTIC MOVIE

📊 BERT SENTIMENT ANALYSIS RESULT:
   Review Quality: Excellent/Masterpiece 😍
   Final Recommendation: Must Watch

--------------------------------------------------
Movie ka naam likhen